In [2]:
import pandas as pd
import numpy as np


In [3]:
df = pd.read_csv('../data/split/engine_split.csv')
df.head(5)

,engine_id,role,fold
0,1,dev,1.0
1,2,dev,4.0
2,3,dev,1.0
3,4,dev,3.0
4,5,dev,4.0


In [ ]:
df = df[df['role']=='dev']

array([  1,   2,   3,   4,   5,   6,   7,  10,  11,  12,  14,  15,  16,
        17,  19,  20,  21,  22,  23,  24,  25,  26,  27,  28,  29,  30,
        31,  32,  33,  34,  35,  36,  38,  39,  40,  41,  42,  43,  44,
        45,  47,  49,  51,  52,  53,  54,  56,  57,  58,  59,  60,  61,
        63,  65,  66,  67,  68,  70,  71,  73,  75,  77,  78,  79,  80,
        81,  82,  83,  86,  88,  89,  91,  92,  93,  94,  96,  97,  98,
        99, 100])

### 교차검증 결과
dev 80대 기준 k = 5로 나누었을 떄 학습엔진 64대, 검증엔진 16대


| Fold | 엔진 수 | RMSE | PHM 점수 | 늦은 예측 비율 |
|---|---|---|---|---|
| 0 | 16 | 13.55 | 3.56 | 67% |
| 1 | 16 | 12.35 | 2.86 | 66% |
| 2 | 16 | 13.31 | 3.47 | 67% |
| 3 | 16 | 14.74 | 4.10 | 64% |
| 4 | 16 | 12.51 | 2.37 | 48% |
| **평균 ± 표준편차** | 80 | **13.29 ± 0.96** | **3.27 ± 0.67** | **62.4 ± 8.1%** |

- **늦은 예측이 절반을 넘음.** fold 0~3에서 64~67%. 늦은 예측은 실제보다 RUL을 길게 예측한 경우로, 정비가 고장보다 늦어질 수 있는 위험한 방향임. PHM 점수가 늦은 예측에 더 큰 벌점을 주기 때문에, 이 비율을 낮추는 게 RMSE를 낮추는 것보다 우선
- **fold 4만 늦은 예측이 48%로 낮고 PHM도 가장 좋음.** 이 fold에 들어간 엔진들의 수명 분포가 다른 fold와 다른지 확인해보아야함.

-> 
- **fold4에서 왜 늦은 예측 비율이 감소했나?** 4폴드에서 엔진들의 평균 수명이 기존 평균 수명 보다 30~40정도 길었음. 실제로 수명과 사이클의 상관관계가 -0.76인것을 볼 떄 수명이 길수록 이른 예측이 많아지고 수명이 짧을수록 지연예측이 많아진다는 사실을 확인함. 그러나 실시간 모니터링에서는 언제 고장날 지 알 수 없기 떄문에 알람 기준을 수명이 짧은 엔진들까지 놓치지 않도록 정하고자함.


### T 후보 
L = 정비에 필요한 사이클
리드타임 = 알람이 울릴 떄 남은 실제 RUL
=> L보다 T가 커야함

T = 15, 20, 25, 30, 35, 40, 45, 50, 55, 60
L = 10,20,30 (수명의 5%~15% 정도만 사용)

놓친 엔진 수 = T < L
평균 낭비 수명 = T - L
리드_min = 엔진별 리드타임의 최소값
리드_p10 = 하위 10% 리드값
리드_중앙값 = 평균 수명을 가진 엔진들의 리드타임

T를 정하는 기준 : 놓쳣을 때의 비용이 일찍 예측했을 때의 비용보다 훨씬 크므로 놓치는 엔진이 없어야한다.     
L을 정하는 기준 : 어떤 현장이냐에 따라 다르기 때문에 임의로 정함. 수명의 5%~10%만 사용하므로 20을 가정함. 
<br> 

L = 20

|T|놓친엔진수|폴드별최대놓침|리드타임평균|리드타임하위10%|리드타임중앙값|평균낭비수명|깜빡임_엔진수|깜빡임최대횟수|
|-|-------|----------|---------|-------------|--------|---------|----------|---------|
|15|72|15|7|10|13|2.6|18|2|
|20|39|10|9|13.9|20|4.8|36|4|
|25|12|3|16|19|25|7.1|39|3|
|30|3|2|18|23|30|11.8|44|5|
|35|1|1|19|25|36|16.3|43|4|
|40|0|0|25|30|40|20.4|37|5|
|45|0|0|29|35|45.5|26.7|57|4|
|50|0|0|34|39.9|51|32|55|5|
|55|0|0|34|42.9|56|37.3|55|6|
|60|0|0|38|45|60|40.7|56|7|

*=> 고장했을 떄의 손실 비용이 이른 예측했을 떄의 잔여수명보다 크다고 가정하므로, 또한 모델이 늦게 예측하는 경향을 보이므로 놓친 엔진 수가 없을 떄를 기준으로 T를 잡음.*
L이 20일 때 T의 값은 40이상일 때 놓친 엔진 수가 0임. RUL이 줄어듬에 따라 오차도 줄어들기 때문에, 너무 높은 T를 잡는건 오차가 크고, 낭비 수명이 크기 떄문에 가장 작은 값을 선택함. 따라서 T = 40을 기준으로 잡음


### 최종 결정안

- T (알람이 울리는 예측 RUL) : 40   
    - 고장 손실 비용이 낭비 수명 비용보다 훨씬 크다고 가정. 잔차(실제 − 예측)가 전 구간에서 음수로 늦은 예측 경향이 있어 놓침 방지를 최우선으로 함   
    - dev 80대 OOF 분석에서 L = 20일 때 놓친 엔진 수가 0인 T는 40 이상. T가 커지면 놓침은 그대로 0이고 낭비 수명만 늘어나므로 최소값 40으로 설정   
    - 민감도: L = 10이면 여백 15, L = 30이면 여백 20 필요  (L ≤ 30 범위에서만 확인)   
- L (정비에 필요한 사이클 시간) : 20   
    - 이유 : L은 회사마다, 장비마다 다르기 때문에 지표로 기준을 삼기 어려워 dev 80대의 평균 수명의 약 10%로 정함    
- 알람 단계 및 기준 : 30일 때 리드타임 최소값이 L보다 작게 나옴 -> 30일 때는 일부 엔진의 실제 RUL은 이미 20이하.   
    - 1단계 알람(T값) : RUL = 40 - "정비 준비" 
    - 2단계 알람(L값+𝜶) : RUL = 30 - "긴급 : 정비 필요"
    - 1단계 알람 후 색깔 변경 - 2단계 전까지 유지
    - 2단계 알람 후 같은 계열의 진한 색으로 색깔 변경 - 끝까지 유지
- 정비 우선순위
    - 알람 단계 : 2단계가 우선
    - RUL기준 : RUL이 가장 엔진을 우선으로 정비
    - 알람단계가 2단계고 예측 RUL이 모두 동일할 때 : 더 오래 대기한 순서

In [6]:
fold_df = pd.read_csv('../data/oof/oof_dev.csv')
fold_df

,engine_id,fold,cycle,RUL,pred,warmup
0,1,1.0,1,191,116.388955,True
1,1,1.0,2,190,103.247228,True
2,1,1.0,3,189,115.706307,True
3,1,1.0,4,188,117.328961,True
4,1,1.0,5,187,112.200740,True
...,...,...,...,...,...,...
16385,100,1.0,196,4,3.069483,False
16386,100,1.0,197,3,3.159951,False
16387,100,1.0,198,2,1.178184,False
16388,100,1.0,199,1,0.981431,False


In [11]:
fold_df[fold_df['fold'] != 1.0].describe()

,engine_id,fold,cycle,RUL,pred
count,13216.000000,13216.000000,13216.000000,13216.000000,13216.000000
mean,51.281023,2.279585,108.959367,107.959367,82.144046
std,29.767601,1.499096,68.932963,68.932963,35.935468
min,2.000000,0.000000,1.000000,0.000000,-1.485435
25%,23.000000,0.000000,52.000000,51.000000,53.814981
50%,53.000000,3.000000,104.000000,103.000000,99.019520
75%,79.000000,4.000000,157.000000,156.000000,111.652324
max,99.000000,4.000000,341.000000,340.000000,125.241632


In [8]:
fold_df[fold_df['fold'] == 4.0].describe()

,engine_id,fold,cycle,RUL,pred
count,3564.000000,3564.0,3564.000000,3564.000000,3564.000000
mean,45.035915,4.0,115.752806,114.752806,82.237464
std,25.980224,0.0,71.148537,71.148537,36.316113
min,2.000000,4.0,1.000000,0.000000,-1.441958
25%,21.000000,4.0,56.000000,55.000000,51.590048
50%,47.000000,4.0,112.000000,111.000000,100.453729
75%,67.000000,4.0,168.000000,167.000000,111.905030
max,86.000000,4.0,313.000000,312.000000,121.401294
